# Text BiLRP — sentence similarity with BERT

> **Extension beyond the paper.** BiLRP (Eberle, Büttner, Kräutli, Müller, Valleriani,
> Montavon, TPAMI 2022) demonstrates token-sequence matching only on a toy digits task
> (§4: two 6-digit sequences as synthetic 10-dim vectors through a small MLP, ground truth
> = number of matching digits) — the bipartite element-matching format this notebook
> inherits. It contains **no BERT / transformer / real-text experiment**, so everything
> language-specific below is our design, not the paper's:
>
> * **Encoder** `phi(x)` = mean-pooled BERT `last_hidden_state`, un-normalized. BiLRP
>   decomposes the **dot product**; the **cosine** map is recovered exactly afterwards by
>   dividing by `||phi(a)|| * ||phi(b)||` (R is bilinear in the two embeddings). Special
>   tokens ([CLS]/[SEP]) participate in the pooling (they are part of phi) and are filtered
>   only from the displayed matrix.
> * **Rules**: epsilon on linears + the AttnLRP attention treatment
>   (`attn='attnlrp'`: Jacobian softmax + bilinear-epsilon matmuls; Achtibat et al., ICML
>   2024) — passthrough softmax measurably under-attributes transformer attention.
> * **Conservation caveat**: with `inputs_embeds`, BERT still *adds position/token-type
>   embeddings internally*; that branch is a constant added to the input, a bias, and a bias
>   keeps its share of the relevance. The printed conservation ratio
>   quantifies exactly how much relevance reaches the input tokens — expect a few percent
>   for a 12-layer biased transformer (Prop. 1 assumes zero biases; BERT has 70+ biased
>   linears whose absorption compounds, squared across the two BiLRP branches — consistent
>   with the ~5–10% single-pass LRP embedding mass we measure on transformers). The
>   *relative* token-pair structure is the signal.

In [ ]:
import sys
sys.path.insert(0, '..')          # examples/showcase: _common
sys.path.insert(0, '../../..')    # repo root: autoLRP (or `pip install -e .`)
import math
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel

import autoLRP as autolrp
from autoLRP import LRPConfig
import _common  # noqa

MODEL_ID = 'bert-base-uncased'
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID).eval()

class MeanPoolEncoder(torch.nn.Module):
    """phi(x) = mean-pooled last_hidden_state, UN-normalized. BiLRP
    decomposes the dot product <phi(a), phi(b)>; the cosine version is
    recovered exactly afterwards by dividing R (which is bilinear in the
    two embeddings) by ||phi(a)||*||phi(b)|| — cleaner than baking a
    detached norm into the graph, whose native-VJP scalar division would
    rescale relevance by 1/||phi|| per side and wreck the conservation
    bookkeeping."""
    def __init__(self, m):
        super().__init__()
        self.model = m
    def forward(self, x):
        return self.model(inputs_embeds=x).last_hidden_state.mean(dim=1)

encoder = MeanPoolEncoder(model)

## BiLRP via `autolrp.bilrp`

We pass the BERT word-embedding tensors directly to the encoder; the per-side reducer sums relevance over the embedding dim to yield one scalar per token.

In [ ]:
SPECIAL = {'[CLS]', '[SEP]', '[PAD]'}

# epsilon (~LRP-0, the conservative family of the paper's Prop. 1) +
# AttnLRP attention treatment (Achtibat et al. 2024): Jacobian softmax,
# bilinear-epsilon attention matmuls.
bilrp_cfg = LRPConfig(attn='attnlrp')

def text_bilrp(s1, s2, n_dims=64):
    ids1 = tok(s1, return_tensors='pt')
    ids2 = tok(s2, return_tensors='pt')
    emb1 = model.embeddings(ids1['input_ids']).detach()
    emb2 = model.embeddings(ids2['input_ids']).detach()
    t1 = tok.convert_ids_to_tokens(ids1['input_ids'][0])
    t2 = tok.convert_ids_to_tokens(ids2['input_ids'][0])
    k1 = [i for i, t in enumerate(t1) if t not in SPECIAL]
    k2 = [i for i, t in enumerate(t2) if t not in SPECIAL]

    with torch.no_grad():
        pa, pb = encoder(emb1), encoder(emb2)
        na, nb = float(pa.norm()), float(pb.norm())
        sim = float((pa * pb).sum()) / (na * nb)          # cosine

    torch.manual_seed(42)                     # reproducible JL projection
    R, dot_proj = autolrp.bilrp(
        encoder, emb1, emb2,
        n_dims=n_dims,
        reduce_each=lambda r: r.sum(dim=-1),  # (B, seq_len)
        config=bilrp_cfg,
        return_similarity=True,
    )
    # Exact cosine rescaling: R is bilinear in the two (projected)
    # embeddings, so dividing by ||phi_a||*||phi_b|| turns the
    # dot-product decomposition into the cosine decomposition.
    R = R / (na * nb)
    cos_proj = dot_proj / (na * nb)
    # Conservation (paper Prop. 1) on the FULL matrix, before display
    # filtering. The shortfall vs 1.0 is what biases keep: linear biases,
    # BERT's internal position/token-type embedding branches, and the
    # non-conservative Jacobian softmax.
    ratio = float(R.sum()) / (cos_proj + 1e-12)
    print(f'  cos={sim:.3f}  projected target={cos_proj:.4f}  '
          f'sum R={float(R.sum()):.4f}  conservation ratio={ratio:.3f}')
    R = R[k1][:, k2].numpy()                  # display filter
    return R, [t1[i] for i in k1], [t2[i] for i in k2], sim

In [ ]:
def plot_bipartite(R, labels1, labels2, title, top_k=15):
    n1, n2 = len(labels1), len(labels2)
    fig, ax = plt.subplots(figsize=(8, max(4, max(n1, n2) * 0.5)))
    y1 = np.linspace(0, 1, n1)
    y2 = np.linspace(0, 1, n2)
    flat = np.abs(R).flatten()
    thresh = np.sort(flat)[-min(top_k, len(flat))]
    rmax = np.abs(R).max() + 1e-10
    for i in range(n1):
        for j in range(n2):
            if abs(R[i, j]) >= thresh:
                c = '#d73027' if R[i, j] > 0 else '#4575b4'
                ax.plot([0, 1], [y1[i], y2[j]], color=c,
                        linewidth=5 * abs(R[i, j]) / rmax, alpha=0.7)
    for i, l in enumerate(labels1):
        ax.text(-0.02, y1[i], l + ' •', ha='right', va='center', fontsize=11)
    for j, l in enumerate(labels2):
        ax.text(1.02, y2[j], '• ' + l, ha='left', va='center', fontsize=11)
    ax.set_xlim(-0.3, 1.3); ax.set_ylim(-0.05, 1.05); ax.axis('off')
    ax.set_title(title); plt.tight_layout(); plt.show()

In [ ]:
pairs = [
    ('The quick brown fox jumps over the lazy dog',
     'The fast auburn fox leapt over a sleepy canine'),
    ('Artificial intelligence is transforming industries',
     'Industries are being revolutionized by advances in AI technology'),
]

for s1, s2 in pairs:
    R, l1, l2, sim = text_bilrp(s1, s2, n_dims=64)
    plot_bipartite(R, l1, l2, f'BERT BiLRP, sim = {sim:.3f}')